# Notebook 3: VGG16 — Embryo Phase Classification with ECL Loss

**Task:** 16-class frame-level classification of embryo developmental phases
**Strategy:** Transfer Learning — ImageNet pretrained backbone frozen, classification head trained
**Model:** VGG16
**Loss:** Embryo Composite Loss (ECL)
**Dataset:** Gomez et al. 2022 — 704 embryos, 297,428 labeled frames, 16 phases


In [1]:
import os

_candidates = [
    '/kaggle/input/datasets/abhishekbuddiga06/embryo-dataset',
    '/kaggle/input/embryo-dataset',
    '/kaggle/input/embryo_dataset',
]
_base = next((p for p in _candidates if os.path.isdir(p)), None)
if _base is None:
    raise FileNotFoundError('embryo-dataset not found in /kaggle/input/')

_img = [os.path.join(_base, 'embryo_dataset', 'embryo_dataset'),
        os.path.join(_base, 'embryo_dataset')]
_ann = [os.path.join(_base, 'embryo_dataset_annotations', 'embryo_dataset_annotations'),
        os.path.join(_base, 'embryo_dataset_annotations')]

IMAGES_ROOT_OVERRIDE = next((p for p in _img if os.path.isdir(p)), None)
ANN_ROOT_OVERRIDE    = next((p for p in _ann if os.path.isdir(p)), None)
PROJECT_ROOT         = '/kaggle/working'

IMAGES_ROOT      = IMAGES_ROOT_OVERRIDE
ANNOTATIONS_ROOT = ANN_ROOT_OVERRIDE

print('Images     :', IMAGES_ROOT)
print('Annotations:', ANNOTATIONS_ROOT)

Images     : /kaggle/input/datasets/abhishekbuddiga06/embryo-dataset/embryo_dataset/embryo_dataset
Annotations: /kaggle/input/datasets/abhishekbuddiga06/embryo-dataset/embryo_dataset_annotations/embryo_dataset_annotations


### Utilities — seed, device, checkpoint helpers

In [2]:
import os
import random
import numpy as np
import torch

def set_seed(seed: int=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def get_device() -> torch.device:
    if torch.cuda.is_available():
        device = torch.device('cuda')
        print(f'Using GPU: {torch.cuda.get_device_name(0)}')
    else:
        device = torch.device('cpu')
        print('Using CPU (no GPU available)')
    return device

def load_checkpoint(model, checkpoint_path: str, device: torch.device) -> dict:
    ckpt = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    print(f'Loaded checkpoint from {checkpoint_path} (epoch={ckpt.get('epoch', '?')}, val_acc={ckpt.get('val_acc', '?'):.4f})')
    return ckpt

def save_history(history: dict, path: str='checkpoints/history.npz'):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    np.savez(path, **{k: np.array(v) for k, v in history.items()})
    print(f'History saved to {path}')

def load_history(path: str='checkpoints/history.npz') -> dict:
    data = np.load(path)
    return {k: data[k].tolist() for k in data.files}

### Dataset — frame index, splits, transforms, EmbryoDataset

In [3]:
import os
import glob
import pandas as pd
import numpy as np
from PIL import Image, ImageFile
from torch.utils.data import Dataset
import torchvision.transforms as T
ImageFile.LOAD_TRUNCATED_IMAGES = True
PHASES = ['tPB2', 'tPNa', 'tPNf', 't2', 't3', 't4', 't5', 't6', 't7', 't8', 't9+', 'tM', 'tSB', 'tB', 'tEB', 'tHB']
PHASE_DISPLAY = ['pPB2', 'pPNa', 'pPNf', 'p2', 'p3', 'p4', 'p5', 'p6', 'p7', 'p8', 'p9+', 'pM', 'pSB', 'pB', 'pEB', 'pHB']
PHASE_TO_IDX = {p: i for i, p in enumerate(PHASES)}
IDX_TO_PHASE = {i: p for i, p in enumerate(PHASE_DISPLAY)}
NUM_CLASSES = len(PHASES)

def build_frame_index(images_root: str, annotations_root: str) -> pd.DataFrame:
    records = []
    annotation_files = glob.glob(os.path.join(annotations_root, '**', '*.csv'), recursive=True)
    annotation_files += glob.glob(os.path.join(annotations_root, '*.csv'))
    ann_map = {}
    for f in annotation_files:
        embryo_id = os.path.basename(f).replace('_phases.csv', '').replace('.csv', '')
        ann_map[embryo_id] = f
    embryo_dirs = sorted([d for d in os.listdir(images_root) if os.path.isdir(os.path.join(images_root, d))])
    skipped = 0
    for embryo_id in embryo_dirs:
        embryo_dir = os.path.join(images_root, embryo_id)
        ann_path = ann_map.get(embryo_id)
        if ann_path is None:
            for key in ann_map:
                if embryo_id in key or key in embryo_id:
                    ann_path = ann_map[key]
                    break
        if ann_path is None:
            skipped += 1
            continue
        try:
            ann = pd.read_csv(ann_path, header=None, names=['phase', 'start', 'end'])
        except Exception:
            skipped += 1
            continue
        ann = ann[ann['phase'].isin(PHASES)].reset_index(drop=True)
        if ann.empty:
            skipped += 1
            continue
        frame_files = sorted(glob.glob(os.path.join(embryo_dir, '*.jpeg')) + glob.glob(os.path.join(embryo_dir, '*.jpg')))
        if not frame_files:
            skipped += 1
            continue
        frame_label = {}
        for _, row in ann.iterrows():
            phase = row['phase']
            if phase not in PHASE_TO_IDX:
                continue
            label_idx = PHASE_TO_IDX[phase]
            for fi in range(int(row['start']), int(row['end']) + 1):
                frame_label[fi] = label_idx
        for frame_path in frame_files:
            fname = os.path.splitext(os.path.basename(frame_path))[0]
            import re as _re
            run_match = _re.search('RUN(\\d+)', fname, _re.IGNORECASE)
            if run_match:
                frame_idx = int(run_match.group(1))
            else:
                digits = _re.findall('\\d+', fname)
                if not digits:
                    continue
                frame_idx = int(digits[-1])
            if frame_idx not in frame_label:
                continue
            label_idx = frame_label[frame_idx]
            records.append({'embryo_id': embryo_id, 'frame_path': frame_path, 'label_idx': label_idx, 'phase_name': PHASES[label_idx]})
    if skipped > 0:
        print(f'[dataset] Warning: skipped {skipped} embryos (missing/unreadable annotations)')
    df = pd.DataFrame(records)
    print(f'[dataset] Built index: {len(df)} labeled frames from {df['embryo_id'].nunique()} embryos')
    return df

def split_by_embryo(df: pd.DataFrame, train_frac=0.7, val_frac=0.15, seed=42):
    embryo_ids = df['embryo_id'].unique()
    rng = np.random.default_rng(seed)
    rng.shuffle(embryo_ids)
    n = len(embryo_ids)
    n_train = int(n * train_frac)
    n_val = int(n * val_frac)
    train_ids = set(embryo_ids[:n_train])
    val_ids = set(embryo_ids[n_train:n_train + n_val])
    test_ids = set(embryo_ids[n_train + n_val:])
    train_df = df[df['embryo_id'].isin(train_ids)].reset_index(drop=True)
    val_df = df[df['embryo_id'].isin(val_ids)].reset_index(drop=True)
    test_df = df[df['embryo_id'].isin(test_ids)].reset_index(drop=True)
    print(f'[dataset] Split — Train: {len(train_df)} frames ({len(train_ids)} embryos) | Val: {len(val_df)} frames ({len(val_ids)} embryos) | Test: {len(test_df)} frames ({len(test_ids)} embryos)')
    return (train_df, val_df, test_df)

def get_class_weights(train_df: pd.DataFrame) -> list:
    counts = train_df['label_idx'].value_counts().sort_index()
    counts = counts.reindex(range(NUM_CLASSES), fill_value=1)
    weights = 1.0 / np.sqrt(counts.values)  # sqrt reduces max ratio 527x -> ~23x
    weights = weights / weights.sum() * NUM_CLASSES
    return weights.tolist()

def get_train_transform(img_size=224):
    return T.Compose([T.Resize((img_size, img_size)), T.RandomHorizontalFlip(), T.RandomRotation(degrees=10), T.ColorJitter(brightness=0.2, contrast=0.2), T.ToTensor(), T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])

def get_val_transform(img_size=224):
    return T.Compose([T.Resize((img_size, img_size)), T.ToTensor(), T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])

class EmbryoDataset(Dataset):

    def __init__(self, frame_df: pd.DataFrame, transform=None):
        self.df = frame_df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['frame_path']).convert('L')
        img = img.convert('RGB')
        if self.transform:
            img = self.transform(img)
        label = int(row['label_idx'])
        return (img, label)

### Models — MobileNetV3 / InceptionV3 / VGG16 / VGG19 builders

In [4]:
import torch
import torch.nn as nn
import torchvision.models as models

def build_model(model_name: str='mobilenet', num_classes: int=NUM_CLASSES, dropout: float=0.3, pretrained: bool=True) -> nn.Module:
    model_name = model_name.lower().strip()
    if model_name == 'mobilenet':
        return _build_mobilenet(num_classes, dropout, pretrained)
    elif model_name == 'inception':
        return _build_inception(num_classes, dropout, pretrained)
    elif model_name == 'vgg16':
        return _build_vgg16(num_classes, dropout, pretrained)
    elif model_name == 'vgg19':
        return _build_vgg19(num_classes, dropout, pretrained)
    else:
        raise ValueError(f'Unknown model: {model_name}. Choose from: mobilenet, inception, vgg16, vgg19')

def _build_mobilenet(num_classes: int, dropout: float, pretrained: bool) -> nn.Module:
    weights = models.MobileNet_V3_Small_Weights.IMAGENET1K_V1 if pretrained else None
    model = models.mobilenet_v3_small(weights=weights)
    in_features = model.classifier[-1].in_features
    model.classifier[-1] = nn.Sequential(nn.Dropout(p=dropout), nn.Linear(in_features, num_classes))
    return model

def _build_inception(num_classes: int, dropout: float, pretrained: bool) -> nn.Module:
    weights = models.Inception_V3_Weights.IMAGENET1K_V1 if pretrained else None
    model = models.inception_v3(weights=weights, aux_logits=False)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(nn.Dropout(p=dropout), nn.Linear(in_features, num_classes))
    return model

def _build_vgg16(num_classes: int, dropout: float, pretrained: bool) -> nn.Module:
    weights = models.VGG16_Weights.IMAGENET1K_V1 if pretrained else None
    model = models.vgg16(weights=weights)
    model.classifier = nn.Sequential(nn.Linear(25088, 4096), nn.ReLU(inplace=True), nn.Dropout(p=dropout), nn.Linear(4096, 1024), nn.ReLU(inplace=True), nn.Dropout(p=dropout), nn.Linear(1024, num_classes))
    return model

def _build_vgg19(num_classes: int, dropout: float, pretrained: bool) -> nn.Module:
    weights = models.VGG19_Weights.IMAGENET1K_V1 if pretrained else None
    model = models.vgg19(weights=weights)
    model.classifier = nn.Sequential(nn.Linear(25088, 4096), nn.ReLU(inplace=True), nn.Dropout(p=dropout), nn.Linear(4096, 1024), nn.ReLU(inplace=True), nn.Dropout(p=dropout), nn.Linear(1024, num_classes))
    return model

def count_parameters(model: nn.Module) -> dict:
    total = sum((p.numel() for p in model.parameters()))
    trainable = sum((p.numel() for p in model.parameters() if p.requires_grad))
    return {'total': total, 'trainable': trainable}

def get_model_info(model_name: str) -> dict:
    info = {'mobilenet': {'params': 1500000, 'input_size': 224, 'flops_per_image': 56000000, 'expected_accuracy': '~75-82%', 'notes': 'Fastest, most efficient'}, 'inception': {'params': 23800000, 'input_size': 299, 'flops_per_image': 5700000000, 'expected_accuracy': '~82-87%', 'notes': 'Multi-scale, good accuracy'}, 'vgg16': {'params': 138000000, 'input_size': 224, 'flops_per_image': 15500000000, 'expected_accuracy': '~80-85%', 'notes': 'Large, may overfit on small dataset'}, 'vgg19': {'params': 143000000, 'input_size': 224, 'flops_per_image': 19600000000, 'expected_accuracy': '~80-85%', 'notes': 'Largest, deepest, prone to overfitting'}}
    model_name = model_name.lower().strip()
    if model_name not in info:
        raise ValueError(f'Unknown model: {model_name}')
    return info[model_name]
if __name__ == '__main__':
    print('=' * 70)
    print('MODEL INFORMATION')
    print('=' * 70)
    for model_name in ['mobilenet', 'inception', 'vgg16', 'vgg19']:
        print(f'\n{model_name.upper()}')
        print('-' * 70)
        info = get_model_info(model_name)
        print(f'Parameters: {info['params']:,}')
        print(f'Input size: {info['input_size']}×{info['input_size']}')
        print(f'FLOPs/image: {info['flops_per_image']:,}')
        print(f'Expected accuracy: {info['expected_accuracy']}')
        print(f'Notes: {info['notes']}')
        model = build_model(model_name, pretrained=False)
        params = count_parameters(model)
        print(f'Actual parameters: {params['total']:,}')
        dummy = torch.randn(1, 3, info['input_size'], info['input_size'])
        with torch.no_grad():
            out = model(dummy)
        print(f'Forward pass: {dummy.shape} → {out.shape}')
    print('\n' + '=' * 70)

MODEL INFORMATION

MOBILENET
----------------------------------------------------------------------
Parameters: 1,500,000
Input size: 224×224
FLOPs/image: 56,000,000
Expected accuracy: ~75-82%
Notes: Fastest, most efficient
Actual parameters: 1,534,256
Forward pass: torch.Size([1, 3, 224, 224]) → torch.Size([1, 16])

INCEPTION
----------------------------------------------------------------------
Parameters: 23,800,000
Input size: 299×299
FLOPs/image: 5,700,000,000
Expected accuracy: ~82-87%
Notes: Multi-scale, good accuracy


/usr/local/lib/python3.12/dist-packages/torchvision/models/inception.py:43: FutureWarning: The default weight initialization of inception_v3 will be changed in future releases of torchvision. If you wish to keep the old behavior (which leads to long initialization times due to scipy/scipy#11299), please set init_weights=True.
  warnings.warn(


Actual parameters: 21,818,352
Forward pass: torch.Size([1, 3, 299, 299]) → torch.Size([1, 16])

VGG16
----------------------------------------------------------------------
Parameters: 138,000,000
Input size: 224×224
FLOPs/image: 15,500,000,000
Expected accuracy: ~80-85%
Notes: Large, may overfit on small dataset
Actual parameters: 121,690,960
Forward pass: torch.Size([1, 3, 224, 224]) → torch.Size([1, 16])

VGG19
----------------------------------------------------------------------
Parameters: 143,000,000
Input size: 224×224
FLOPs/image: 19,600,000,000
Expected accuracy: ~80-85%
Notes: Largest, deepest, prone to overfitting
Actual parameters: 127,000,656
Forward pass: torch.Size([1, 3, 224, 224]) → torch.Size([1, 16])



### Loss Functions — OFL, SFO, HSL, ECL

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
NUM_CLASSES = 16
PHASE_NAMES = ['pPB2', 'pPNa', 'pPNf', 'p2', 'p3', 'p4', 'p5', 'p6', 'p7', 'p8', 'p9+', 'pM', 'pSB', 'pB', 'pEB', 'pHB']
PHASE_TO_STAGE = [0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 2, 3, 3, 3, 3, 3]
NUM_STAGES = 4

class OrdinalFocalLoss(nn.Module):

    def __init__(self, gamma: float=2.0, lambd: float=0.5, alpha: torch.Tensor=None):
        super().__init__()
        self.gamma = gamma
        self.lambd = lambd
        self.register_buffer('alpha', alpha)

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        C = logits.shape[1]
        device = logits.device
        log_probs = F.log_softmax(logits, dim=1)
        probs = torch.exp(log_probs)
        log_pt = log_probs.gather(1, targets.unsqueeze(1)).squeeze(1)
        pt = probs.gather(1, targets.unsqueeze(1)).squeeze(1)
        focal_weight = (1.0 - pt) ** self.gamma
        L_focal = -focal_weight * log_pt
        if self.alpha is not None:
            alpha_t = self.alpha.to(device).gather(0, targets)
            L_focal = alpha_t * L_focal
        class_idx = torch.arange(C, device=device).float()
        targets_f = targets.float().unsqueeze(1)
        distances = torch.abs(class_idx.unsqueeze(0) - targets_f)
        distances = distances / (C - 1)
        L_ordinal = (probs * distances).sum(dim=1)
        loss = L_focal + self.lambd * L_ordinal
        return loss.mean()

class SoftF1OrdinalLoss(nn.Module):

    def __init__(self, lambd: float=0.3, eps: float=1e-07):
        super().__init__()
        self.lambd = lambd
        self.eps = eps

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        C = logits.shape[1]
        N = logits.shape[0]
        device = logits.device
        probs = F.softmax(logits, dim=1)
        one_hot = F.one_hot(targets, num_classes=C).float()
        TP = (probs * one_hot).sum(dim=0)
        FP = (probs * (1.0 - one_hot)).sum(dim=0)
        FN = ((1.0 - probs) * one_hot).sum(dim=0)
        precision = TP / (TP + FP + self.eps)
        recall = TP / (TP + FN + self.eps)
        f1_per_class = 2.0 * precision * recall / (precision + recall + self.eps)
        L_f1 = 1.0 - f1_per_class.mean()
        class_idx = torch.arange(C, device=device).float()
        targets_f = targets.float().unsqueeze(1)
        sq_dist = (class_idx.unsqueeze(0) - targets_f) ** 2 / (C - 1) ** 2
        L_ordinal = (probs * sq_dist).sum(dim=1).mean()
        return L_f1 + self.lambd * L_ordinal

class HierarchicalStageLoss(nn.Module):

    def __init__(self, beta: float=0.5, class_weights: torch.Tensor=None):
        super().__init__()
        self.beta = beta
        phase_to_stage = torch.tensor(PHASE_TO_STAGE, dtype=torch.long)
        self.register_buffer('phase_to_stage', phase_to_stage)
        stage_mask = torch.zeros(NUM_STAGES, NUM_CLASSES)
        for k, s in enumerate(PHASE_TO_STAGE):
            stage_mask[s, k] = 1.0
        self.register_buffer('stage_mask', stage_mask)
        self.ce_phase = nn.CrossEntropyLoss(weight=class_weights)
        self.ce_stage = nn.CrossEntropyLoss()

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        L_phase = self.ce_phase(logits, targets)
        probs = F.softmax(logits, dim=1)
        stage_probs = torch.matmul(probs, self.stage_mask.T)
        stage_log_probs = torch.log(stage_probs + 1e-08)
        stage_targets = self.phase_to_stage[targets]
        L_stage = self.ce_stage(stage_log_probs, stage_targets)
        return L_phase + self.beta * L_stage

class EmbryoCompositeLoss(nn.Module):

    def __init__(self, class_weights: torch.Tensor=None, lambda_ord: float=0.3, lambda_boundary: float=0.5, lambda_rare: float=0.5, boundary_penalty: float=2.0, rare_classes: dict=None):
        super().__init__()
        self.lambda_ord = lambda_ord
        self.lambda_boundary = lambda_boundary
        self.lambda_rare = lambda_rare
        self.boundary_penalty = boundary_penalty
        self.register_buffer('class_weights', class_weights)
        if rare_classes is None:
            rare_classes = {15: 5.0, 4: 2.0, 2: 1.5, 6: 1.2, 7: 1.2}
        self.rare_classes = rare_classes
        self.register_buffer('phase_to_stage', torch.tensor(PHASE_TO_STAGE, dtype=torch.long))

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        C = logits.shape[1]
        N = logits.shape[0]
        device = logits.device
        probs = F.softmax(logits, dim=1)
        log_probs = F.log_softmax(logits, dim=1)
        log_pt = log_probs.gather(1, targets.unsqueeze(1)).squeeze(1)
        if self.class_weights is not None:
            alpha_t = self.class_weights.to(device).gather(0, targets)
            L_ce = -(alpha_t * log_pt)
        else:
            L_ce = -log_pt
        L_ce = L_ce.mean()
        class_idx = torch.arange(C, device=device).float()
        targets_f = targets.float().unsqueeze(1)
        sq_dist = (class_idx.unsqueeze(0) - targets_f) ** 2 / (C - 1) ** 2
        L_ordinal = (probs * sq_dist).sum(dim=1).mean()
        true_stages = self.phase_to_stage[targets]
        pred_stages = self.phase_to_stage.unsqueeze(0).expand(N, -1)
        cross_stage = (pred_stages != true_stages.unsqueeze(1)).float()
        L_boundary = (probs * cross_stage).sum(dim=1).mean() * self.boundary_penalty
        L_rare = torch.tensor(0.0, device=device)
        for cls_idx, boost in self.rare_classes.items():
            mask = targets == cls_idx
            if mask.any():
                p_true = probs[mask, cls_idx]
                fn_penalty = (1.0 - p_true) * boost
                L_rare = L_rare + fn_penalty.mean()
        return L_ce + self.lambda_ord * L_ordinal + self.lambda_boundary * L_boundary + self.lambda_rare * L_rare

def build_loss(loss_type: str='ecl', class_weights=None, **kwargs) -> nn.Module:
    if loss_type == 'ce':
        return nn.CrossEntropyLoss(weight=class_weights)
    elif loss_type == 'ofl':
        return OrdinalFocalLoss(gamma=kwargs.get('gamma', 2.0), lambd=kwargs.get('lambd', 0.5), alpha=class_weights)
    elif loss_type == 'sfo':
        return SoftF1OrdinalLoss(lambd=kwargs.get('lambd', 0.3))
    elif loss_type == 'hsl':
        return HierarchicalStageLoss(beta=kwargs.get('beta', 0.5), class_weights=class_weights)
    elif loss_type == 'ecl':
        return EmbryoCompositeLoss(class_weights=class_weights, lambda_ord=kwargs.get('lambda_ord', 0.3), lambda_boundary=kwargs.get('lambda_boundary', 0.5), lambda_rare=kwargs.get('lambda_rare', 0.5), boundary_penalty=kwargs.get('boundary_penalty', 2.0))
    else:
        raise ValueError(f"Unknown loss: '{loss_type}'. Choose from: ce, ofl, sfo, hsl, ecl")

### Evaluation — inference, plots, classification report

In [6]:
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, f1_score
PLOTS_DIR = 'plots'

@torch.no_grad()
def predict(model: nn.Module, loader: DataLoader, device: torch.device) -> tuple[np.ndarray, np.ndarray]:
    model.eval()
    all_preds, all_labels = ([], [])
    for imgs, labels in tqdm(loader, desc='Inference', ncols=80):
        imgs = imgs.to(device, non_blocking=True)
        logits = model(imgs)
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.append(preds)
        all_labels.append(labels.numpy())
    return (np.concatenate(all_preds), np.concatenate(all_labels))

def plot_loss_curve(history: dict, save_dir: str=PLOTS_DIR):
    os.makedirs(save_dir, exist_ok=True)
    epochs = range(1, len(history['train_loss']) + 1)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(epochs, history['train_loss'], label='Train Loss', linewidth=2)
    ax.plot(epochs, history['val_loss'], label='Val Loss', linewidth=2, linestyle='--')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('Training vs Validation Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    path = os.path.join(save_dir, 'loss_curve.png')
    plt.savefig(path, dpi=150)
    plt.show()
    plt.close()
    print(f'Saved: {path}')

def plot_accuracy_curve(history: dict, save_dir: str=PLOTS_DIR):
    os.makedirs(save_dir, exist_ok=True)
    epochs = range(1, len(history['train_acc']) + 1)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(epochs, [a * 100 for a in history['train_acc']], label='Train Acc', linewidth=2)
    ax.plot(epochs, [a * 100 for a in history['val_acc']], label='Val Acc', linewidth=2, linestyle='--')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy (%)')
    ax.set_title('Training vs Validation Accuracy')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    path = os.path.join(save_dir, 'accuracy_curve.png')
    plt.savefig(path, dpi=150)
    plt.show()
    plt.close()
    print(f'Saved: {path}')

def plot_lr_curve(history: dict, save_dir: str=PLOTS_DIR):
    os.makedirs(save_dir, exist_ok=True)
    epochs = range(1, len(history['lr']) + 1)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(epochs, history['lr'], linewidth=2, color='orange')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Learning Rate')
    ax.set_title('Learning Rate Schedule')
    ax.set_yscale('log')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    path = os.path.join(save_dir, 'lr_curve.png')
    plt.savefig(path, dpi=150)
    plt.show()
    plt.close()
    print(f'Saved: {path}')

def plot_confusion_matrix(preds: np.ndarray, labels: np.ndarray, save_dir: str=PLOTS_DIR):
    os.makedirs(save_dir, exist_ok=True)
    cm = confusion_matrix(labels, preds, labels=list(range(NUM_CLASSES)))
    cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-08)
    fig, ax = plt.subplots(figsize=(14, 12))
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', xticklabels=PHASE_DISPLAY, yticklabels=PHASE_DISPLAY, linewidths=0.5, ax=ax, vmin=0, vmax=1)
    ax.set_xlabel('Predicted Phase', fontsize=12)
    ax.set_ylabel('True Phase', fontsize=12)
    ax.set_title('Normalized Confusion Matrix', fontsize=14)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    path = os.path.join(save_dir, 'confusion_matrix.png')
    plt.savefig(path, dpi=150)
    plt.show()
    plt.close()
    print(f'Saved: {path}')

def plot_per_class_f1(preds: np.ndarray, labels: np.ndarray, save_dir: str=PLOTS_DIR):
    os.makedirs(save_dir, exist_ok=True)
    f1_scores = f1_score(labels, preds, labels=list(range(NUM_CLASSES)), average=None, zero_division=0)
    colors = ['#2196F3' if s >= 0.5 else '#F44336' for s in f1_scores]
    fig, ax = plt.subplots(figsize=(12, 5))
    bars = ax.bar(PHASE_DISPLAY, f1_scores, color=colors, edgecolor='black', linewidth=0.5)
    ax.axhline(y=np.mean(f1_scores), color='black', linestyle='--', linewidth=1.5, label=f'Mean F1 = {np.mean(f1_scores):.3f}')
    ax.set_ylim(0, 1.05)
    ax.set_xlabel('Development Phase')
    ax.set_ylabel('F1 Score')
    ax.set_title('Per-Class F1 Score')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    plt.xticks(rotation=45, ha='right')
    for bar, score in zip(bars, f1_scores):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01, f'{score:.2f}', ha='center', va='bottom', fontsize=8)
    plt.tight_layout()
    path = os.path.join(save_dir, 'per_class_f1.png')
    plt.savefig(path, dpi=150)
    plt.show()
    plt.close()
    print(f'Saved: {path}')

def plot_class_distribution(train_df, val_df, test_df, save_dir: str=PLOTS_DIR):
    os.makedirs(save_dir, exist_ok=True)
    import pandas as pd

    def counts(df):
        c = df['label_idx'].value_counts().sort_index()
        return c.reindex(range(NUM_CLASSES), fill_value=0).values
    train_c = counts(train_df)
    val_c = counts(val_df)
    test_c = counts(test_df)
    x = np.arange(NUM_CLASSES)
    width = 0.28
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.bar(x - width, train_c, width, label='Train', alpha=0.8)
    ax.bar(x, val_c, width, label='Val', alpha=0.8)
    ax.bar(x + width, test_c, width, label='Test', alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(PHASE_DISPLAY, rotation=45, ha='right')
    ax.set_ylabel('Number of Frames')
    ax.set_title('Class Distribution across Splits')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    path = os.path.join(save_dir, 'class_distribution.png')
    plt.savefig(path, dpi=150)
    plt.show()
    plt.close()
    print(f'Saved: {path}')

def plot_sample_predictions(model: nn.Module, loader: DataLoader, device: torch.device, n_samples: int=16, save_dir: str=PLOTS_DIR):
    os.makedirs(save_dir, exist_ok=True)
    model.eval()
    images, true_labels, pred_labels = ([], [], [])
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            logits = model(imgs)
            preds = logits.argmax(dim=1).cpu()
            images.append(imgs.cpu())
            true_labels.append(labels)
            pred_labels.append(preds)
            if sum((len(x) for x in images)) >= n_samples:
                break
    images = torch.cat(images)[:n_samples]
    true_labels = torch.cat(true_labels)[:n_samples]
    pred_labels = torch.cat(pred_labels)[:n_samples]
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    images = images * std + mean
    images = images.clamp(0, 1)
    cols = 4
    rows = (n_samples + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.5, rows * 3.5))
    axes = axes.flatten()
    for i in range(n_samples):
        img = images[i].permute(1, 2, 0).numpy()
        axes[i].imshow(img[:, :, 0], cmap='gray')
        t = IDX_TO_PHASE[true_labels[i].item()]
        p = IDX_TO_PHASE[pred_labels[i].item()]
        color = 'green' if t == p else 'red'
        axes[i].set_title(f'T: {t}\nP: {p}', fontsize=9, color=color)
        axes[i].axis('off')
    for i in range(n_samples, len(axes)):
        axes[i].axis('off')
    plt.suptitle('Sample Predictions (Green=Correct, Red=Wrong)', fontsize=12)
    plt.tight_layout()
    path = os.path.join(save_dir, 'sample_predictions.png')
    plt.savefig(path, dpi=150)
    plt.show()
    plt.close()
    print(f'Saved: {path}')

def full_evaluation(model: nn.Module, loader: DataLoader, device: torch.device, history: dict, train_df=None, val_df=None, test_df=None, save_dir: str=PLOTS_DIR):
    print('\n=== Evaluation ===')
    preds, labels = predict(model, loader, device)
    report = classification_report(labels, preds, target_names=PHASE_DISPLAY, labels=list(range(NUM_CLASSES)), zero_division=0)
    print('\nClassification Report:')
    print(report)
    os.makedirs(save_dir, exist_ok=True)
    with open(os.path.join(save_dir, 'classification_report.txt'), 'w') as f:
        f.write(report)
    macro_f1 = f1_score(labels, preds, average='macro', zero_division=0)
    acc = (preds == labels).mean()
    print(f'Test Accuracy:  {acc:.4f} ({acc * 100:.2f}%)')
    print(f'Macro F1:       {macro_f1:.4f}')
    print('\nGenerating plots...')
    plot_loss_curve(history, save_dir)
    plot_accuracy_curve(history, save_dir)
    plot_lr_curve(history, save_dir)
    plot_confusion_matrix(preds, labels, save_dir)
    plot_per_class_f1(preds, labels, save_dir)
    plot_sample_predictions(model, loader, device, n_samples=16, save_dir=save_dir)
    if train_df is not None and val_df is not None and (test_df is not None):
        plot_class_distribution(train_df, val_df, test_df, save_dir)
    print(f'\nAll plots saved to: {save_dir}/')
    return {'accuracy': acc, 'macro_f1': macro_f1, 'preds': preds, 'labels': labels}

In [7]:
# Cell 0: GPU / Accelerator Check
import sys, os, time
import torch

n_gpus = torch.cuda.device_count()
device = torch.device('cuda' if n_gpus > 0 else 'cpu')
print(f'Device     : {device}')
print(f'GPU count  : {n_gpus}')
for i in range(n_gpus):
    props = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}   : {torch.cuda.get_device_name(i)}  VRAM={props.total_memory/1e9:.1f}GB  '
          f'SM={props.major}.{props.minor}')
if n_gpus > 1:
    print(f'DataParallel: ENABLED ({n_gpus} GPUs — batch split {n_gpus} ways)')
elif n_gpus == 0:
    print('WARNING: No GPU — training will be very slow on CPU.')
print(f'PyTorch    : {torch.__version__}  CUDA: {torch.version.cuda}')

Device     : cuda
GPU count  : 2
  GPU 0   : Tesla T4  VRAM=15.6GB  SM=7.5
  GPU 1   : Tesla T4  VRAM=15.6GB  SM=7.5
DataParallel: ENABLED (2 GPUs — batch split 2 ways)
PyTorch    : 2.10.0+cu128  CUDA: 12.8


In [8]:
# Cell 1: Configuration

MODEL_NAME    = 'vgg16'
MODEL_DISPLAY = 'VGG16'
IMG_SIZE      = 224
BATCH_SIZE    = 512
NUM_EPOCHS    = 15
PATIENCE      = 4
SEED          = 42
LR            = 5e-4
WEIGHT_DECAY  = 1e-4
DROPOUT       = 0.3

CHECKPOINT_DIR   = '/kaggle/working/checkpoints/vgg16_ecl'
PLOTS_DIR        = '/kaggle/working/plots/vgg16_ecl'

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

import random, numpy as np
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

print(f'Model         : {MODEL_DISPLAY}')
print(f'Image size    : {IMG_SIZE}x{IMG_SIZE}')
print(f'Batch size    : {BATCH_SIZE}')
print(f'Epochs        : {NUM_EPOCHS} (patience={PATIENCE})')
print(f'Images        : {IMAGES_ROOT}')
print(f'Annotations   : {ANNOTATIONS_ROOT}')
print(f'Checkpoints   : {CHECKPOINT_DIR}')


Model         : VGG16
Image size    : 224x224
Batch size    : 512
Epochs        : 15 (patience=4)
Images        : /kaggle/input/datasets/abhishekbuddiga06/embryo-dataset/embryo_dataset/embryo_dataset
Annotations   : /kaggle/input/datasets/abhishekbuddiga06/embryo-dataset/embryo_dataset_annotations/embryo_dataset_annotations
Checkpoints   : /kaggle/working/checkpoints/vgg16_ecl


In [9]:
# Cell 2: Dataset Information and EDA
import matplotlib.pyplot as plt
from PIL import Image

print('Building frame index (this may take 1-2 minutes for 297,428 frames)...')
df = build_frame_index(IMAGES_ROOT, ANNOTATIONS_ROOT)

if len(df) == 0:
    raise RuntimeError(
        'No labeled frames found.\n'
        f'  Images path      : {IMAGES_ROOT}\n'
        f'  Annotations path : {ANNOTATIONS_ROOT}'
    )

counts   = df['label_idx'].value_counts().sort_index()
cnt_list = [counts.get(i, 0) for i in range(NUM_CLASSES)]

print(f'\nDataset Statistics')
print(f'  Total frames  : {len(df):,}')
print(f'  Total embryos : {df["embryo_id"].nunique()}')
print(f'  Classes       : {NUM_CLASSES}')
print(f'  Most common   : {PHASE_DISPLAY[counts.idxmax()]} ({counts.max():,} frames)')
print(f'  Rarest        : {PHASE_DISPLAY[counts.idxmin()]} ({counts.min()} frames)')
print(f'  Imbalance     : {counts.max() // counts.min()}:1')
print()
print(f'  {"Phase":>5}  {"Frames":>8}')
for i, c in enumerate(cnt_list):
    print(f'  {PHASE_DISPLAY[i]:>5}  {c:>8,}')

colors = plt.cm.tab20(np.linspace(0, 1, NUM_CLASSES))
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, sc, ttl in zip(axes,
                        ['linear', 'log'],
                        ['Class Distribution (Linear Scale)', 'Class Distribution (Log Scale)']):
    ax.bar(PHASE_DISPLAY, cnt_list, color=colors, edgecolor='k', linewidth=0.5)
    ax.set_yscale(sc)
    ax.set_title(ttl, fontsize=12)
    ax.set_xlabel('Developmental Phase')
    ax.set_ylabel('Number of Frames')
    ax.tick_params(axis='x', rotation=45)
    ax.grid(axis='y', alpha=0.3)
plt.suptitle(
    f'Embryo Dataset   {len(df):,} Frames   {df["embryo_id"].nunique()} Embryos   527:1 Imbalance',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'dataset_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

Building frame index (this may take 1-2 minutes for 297,428 frames)...
[dataset] Built index: 297428 labeled frames from 704 embryos

Dataset Statistics
  Total frames  : 297,428
  Total embryos : 704
  Classes       : 16
  Most common   : p9+ (51,112 frames)
  Rarest        : pHB (97 frames)
  Imbalance     : 526:1

  Phase    Frames
   pPB2     8,895
   pPNa    43,244
   pPNf     6,793
     p2    29,189
     p3     5,127
     p4    29,177
     p5     8,045
     p6     8,366
     p7    10,531
     p8    32,602
    p9+    51,112
     pM    17,084
    pSB    17,244
     pB    10,387
    pEB    19,535
    pHB        97


In [10]:
# Cell 3: Random Sample Visualization

# One representative frame per class (2 rows x 8 columns)
fig, axes = plt.subplots(2, 8, figsize=(20, 6))
for i, ax in enumerate(axes.flatten()):
    cls_df = df[df['label_idx'] == i]
    if len(cls_df) == 0:
        ax.axis('off')
        continue
    try:
        img = Image.open(cls_df.sample(1, random_state=SEED).iloc[0]['frame_path']).convert('L')
        ax.imshow(img, cmap='gray', aspect='auto')
        ax.set_title(f'{PHASE_DISPLAY[i]}\n({len(cls_df):,} frames)', fontsize=8, fontweight='bold')
    except Exception:
        ax.text(0.5, 0.5, 'read error', ha='center', fontsize=7)
    ax.axis('off')
plt.suptitle('One Representative Frame per Developmental Phase', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'sample_per_class.png'), dpi=150, bbox_inches='tight')
plt.show()

# Random 4x4 grid of frames
samples = df.sample(16, random_state=SEED)
fig, axes = plt.subplots(4, 4, figsize=(12, 12))
for ax, (_, row) in zip(axes.flatten(), samples.iterrows()):
    try:
        img = Image.open(row['frame_path']).convert('L')
        ax.imshow(img, cmap='gray')
        ax.set_title(PHASE_DISPLAY[row['label_idx']], fontsize=9, fontweight='bold')
    except Exception:
        ax.text(0.5, 0.5, 'Error', ha='center')
    ax.axis('off')
plt.suptitle('Random 4x4 Frame Grid', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'random_grid.png'), dpi=150, bbox_inches='tight')
plt.show()


In [11]:
# Cell 4: EmbryoLevel Data Split (No Data Leakage)

train_df, val_df, test_df = split_by_embryo(df, seed=SEED)

# Verify no leakage between splits
tr_ids = set(train_df['embryo_id'])
va_ids = set(val_df['embryo_id'])
te_ids = set(test_df['embryo_id'])
assert not (tr_ids & va_ids), 'Leakage: Train and Val share embryos'
assert not (tr_ids & te_ids), 'Leakage: Train and Test share embryos'
assert not (va_ids & te_ids), 'Leakage: Val and Test share embryos'
print('Embryo-level split verified — no data leakage')

total = len(df)
print()
print(f'{"Split":<8} {"Embryos":>8} {"Frames":>10} {"Fraction":>10}')
print('-' * 40)
for name, sdf in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    print(f'{name:<8} {sdf["embryo_id"].nunique():>8} {len(sdf):>10,} {len(sdf)/total:>9.1%}')
print(f'{"Total":<8} {df["embryo_id"].nunique():>8} {total:>10,} {"100%":>10}')
print()
print(f'Split uses {df["embryo_id"].nunique()} embryos ({len(df):,} frames). Full dataset: 704 embryos / 297,428 frames.')

plot_class_distribution(train_df, val_df, test_df, save_dir=PLOTS_DIR)
plt.show()

[dataset] Split — Train: 208397 frames (492 embryos) | Val: 44167 frames (105 embryos) | Test: 44864 frames (107 embryos)
Embryo-level split verified — no data leakage

Split     Embryos     Frames   Fraction
----------------------------------------
Train         492    208,397     70.1%
Val           105     44,167     14.8%
Test          107     44,864     15.1%
Total         704    297,428       100%

Split uses 704 embryos (297,428 frames). Full dataset: 704 embryos / 297,428 frames.
Saved: /kaggle/working/plots/vgg16_ecl/class_distribution.png


In [12]:
# Cell 5: DataLoaders
from torch.utils.data import DataLoader

train_ds = EmbryoDataset(train_df, transform=get_train_transform(IMG_SIZE))
val_ds   = EmbryoDataset(val_df,   transform=get_val_transform(IMG_SIZE))
test_ds  = EmbryoDataset(test_df,  transform=get_val_transform(IMG_SIZE))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True)

print(f'Train loader : {len(train_ds):>8,} frames  {len(train_loader)} batches  BS={BATCH_SIZE}')
print(f'Val loader   : {len(val_ds):>8,} frames  {len(val_loader)} batches')
print(f'Test loader  : {len(test_ds):>8,} frames  {len(test_loader)} batches')

imgs, labels = next(iter(train_loader))
print(f'Batch shape  : {tuple(imgs.shape)}')

Train loader :  208,397 frames  408 batches  BS=512
Val loader   :   44,167 frames  87 batches
Test loader  :   44,864 frames  88 batches
Batch shape  : (512, 3, 224, 224)


In [13]:
# Cell 6: Model Setup — VGG16 Transfer Learning + DataParallel
#
# Order matters:
#   1. build_model()          — instantiate pretrained VGG16
#   2. .to(device)            — move weights to GPU 0
#   3. freeze backbone        — disable gradients on features (saves VRAM & compute)
#   4. nn.DataParallel(model) — split each batch across all available GPUs
#
model = build_model(model_name=MODEL_NAME, dropout=DROPOUT, pretrained=True)
model = model.to(device)

# Step 3: freeze all backbone except last conv block (features[-7:])
# Last block adapts ImageNet features to embryo microscopy domain
for param in model.features.parameters():
    param.requires_grad = False
for param in model.features[-7:].parameters():  # unfreeze last conv block
    param.requires_grad = True

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params    = total_params - trainable_params
info = get_model_info(MODEL_NAME)

# Step 4: DataParallel — wraps model so each GPU processes BATCH_SIZE//n_gpus samples
if torch.cuda.device_count() > 1:
    model = torch.nn.DataParallel(model)
    print(f'DataParallel : {torch.cuda.device_count()} GPUs  '
          f'effective batch/GPU = {BATCH_SIZE // torch.cuda.device_count()}')

print(f'Model              : {MODEL_DISPLAY}')
print(f'Total parameters   : {total_params:,}')
print(f'Frozen (features)  : {frozen_params:,}  (ImageNet conv backbone)')
print(f'Trainable (classif): {trainable_params:,}  (FC head only)')
print(f'FLOPs per image    : {info["flops_per_image"]/1e9:.1f}B')

dummy = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(device)
with torch.no_grad():
    out = model(dummy)
assert out.shape == (2, 16), f'Wrong output shape: {out.shape}'
print(f'Forward pass       : {tuple(dummy.shape)} -> {tuple(out.shape)}  OK')
del dummy, out
torch.cuda.empty_cache()

Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:02<00:00, 188MB/s]


DataParallel : 2 GPUs  effective batch/GPU = 256
Model              : VGG16
Total parameters   : 121,690,960
Frozen (features)  : 7,635,264  (ImageNet conv backbone)
Trainable (classif): 114,055,696  (FC head only)
FLOPs per image    : 15.5B
Forward pass       : (2, 3, 224, 224) -> (2, 16)  OK


In [14]:
# Cell 7: Embryo Composite Loss (ECL) + Optimizer
#
# ECL = L_ce + 0.3 * L_ordinal + 0.5 * L_boundary + 0.5 * L_rare
#
# L_ce      : weighted cross-entropy (inverse-frequency class weights)
# L_ordinal : quadratic penalty for predicting phases far from the true phase
# L_boundary: extra penalty when prediction crosses a biological stage boundary
# L_rare    : false-negative penalty for rare classes (pHB, p3, pPNf, p5, p6)
#
import torch.nn.functional as F

weights       = get_class_weights(train_df)
class_weights = torch.tensor(weights, dtype=torch.float32).to(device)

criterion = EmbryoCompositeLoss(
    class_weights=class_weights,
    lambda_ord=0.3,
    lambda_boundary=0.5,
    lambda_rare=0.2,
    boundary_penalty=2.0,
).to(device)

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS, eta_min=1e-6
)

print('ECL Loss: L = L_ce + 0.3*L_ordinal + 0.5*L_boundary + 0.5*L_rare')
print(f'Optimizer  : AdamW  lr={LR}  weight_decay={WEIGHT_DECAY}')
print(f'Scheduler  : CosineAnnealingLR  T_max={NUM_EPOCHS}  eta_min=1e-6')
print(f'Parameters : {trainable_params:,} (head only)')

ECL Loss: L = L_ce + 0.3*L_ordinal + 0.5*L_boundary + 0.5*L_rare
Optimizer  : AdamW  lr=0.0005  weight_decay=0.0001
Scheduler  : CosineAnnealingLR  T_max=15  eta_min=1e-6
Parameters : 114,055,696 (head only)


In [15]:
from tqdm import tqdm
from sklearn.metrics import f1_score
import torch.nn.functional as F

LATEST_CKPT  = os.path.join(CHECKPOINT_DIR, 'latest_checkpoint.pth')
BEST_F1_CKPT = os.path.join(CHECKPOINT_DIR, 'best_f1_model.pth')

history = {
    'train_loss': [], 'val_loss': [],
    'train_acc':  [], 'val_acc':  [],
    'train_f1':   [], 'val_f1':   [],
    'lr': [],
    'ecl_ce': [], 'ecl_ordinal': [], 'ecl_boundary': [], 'ecl_rare': [],
}

best_val_f1 = 0.0
no_improve  = 0
start_time  = time.time()
use_amp     = device.type == 'cuda'
scaler      = torch.amp.GradScaler('cuda') if use_amp else None

print(f'Training {MODEL_DISPLAY} with ECL Loss')
print(f'Backbone frozen | {trainable_params:,} head params | AMP={use_amp}')
print(f'{"Epoch":>5}  {"TrLoss":>7}  {"TrAcc":>6}  {"TrF1":>6}  {"VaLoss":>7}  {"VaAcc":>6}  {"VaF1":>6}  {"CE":>6}  {"Ord":>6}  {"Bnd":>6}  {"Rare":>6}  {"LR":>8}  {"Time":>5}')
print('-' * 110)

for epoch in range(1, NUM_EPOCHS + 1):
    t0     = time.time()
    cur_lr = optimizer.param_groups[0]['lr']

    # ── Train ──────────────────────────────────────────────────────────────────
    model.train()
    tr_loss = tr_correct = tr_total = 0
    tr_preds, tr_labels = [], []

    for imgs, lbls in tqdm(train_loader, desc=f'Epoch {epoch:02d}/{NUM_EPOCHS} [Train]',
                            leave=False, ncols=90):
        imgs = imgs.to(device, non_blocking=True)
        lbls = lbls.to(device, non_blocking=True)
        optimizer.zero_grad()

        if use_amp:
            with torch.amp.autocast('cuda'):
                logits = model(imgs)
                loss   = criterion(logits, lbls)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(imgs)
            loss   = criterion(logits, lbls)
            loss.backward()
            optimizer.step()

        tr_loss    += loss.item() * imgs.size(0)
        preds       = logits.argmax(1)
        tr_correct += (preds == lbls).sum().item()
        tr_total   += imgs.size(0)
        tr_preds.extend(preds.cpu().numpy())
        tr_labels.extend(lbls.cpu().numpy())

    tr_loss /= tr_total
    tr_acc   = tr_correct / tr_total
    tr_f1    = f1_score(tr_labels, tr_preds, average='macro', zero_division=0)

    # ── Validate ────────────────────────────────────────────────────────────────
    model.eval()
    va_loss = va_correct = va_total = 0
    va_preds, va_labels = [], []

    with torch.no_grad():
        for imgs, lbls in tqdm(val_loader, desc=f'Epoch {epoch:02d}/{NUM_EPOCHS} [Val]  ',
                                leave=False, ncols=90):
            imgs = imgs.to(device, non_blocking=True)
            lbls = lbls.to(device, non_blocking=True)
            logits  = model(imgs)
            loss    = criterion(logits, lbls)
            va_loss += loss.item() * imgs.size(0)
            preds   = logits.argmax(1)
            va_correct += (preds == lbls).sum().item()
            va_total   += imgs.size(0)
            va_preds.extend(preds.cpu().numpy())
            va_labels.extend(lbls.cpu().numpy())

    va_loss /= va_total
    va_acc   = va_correct / va_total
    va_f1    = f1_score(va_labels, va_preds, average='macro', zero_division=0)

    scheduler.step()

    # ── ECL component breakdown (every epoch) ───────────────────────────────────
    model.eval()
    ce_s = ord_s = bnd_s = rare_s = n_b = 0
    with torch.no_grad():
        for imgs, lbls in val_loader:
            if n_b >= 10:
                break
            imgs, lbls = imgs.to(device), lbls.to(device)
            logits = model(imgs)
            C = 16; N = imgs.size(0)
            probs = F.softmax(logits, 1)
            lp    = F.log_softmax(logits, 1)
            lpt   = lp.gather(1, lbls.unsqueeze(1)).squeeze(1)
            at    = criterion.class_weights.gather(0, lbls)
            ce_s   += -(at * lpt).mean().item()
            ci      = torch.arange(C, device=device).float()
            sq      = ((ci.unsqueeze(0) - lbls.float().unsqueeze(1)) ** 2) / ((C - 1) ** 2)
            ord_s  += (probs * sq).sum(1).mean().item()
            ts      = criterion.phase_to_stage[lbls]
            ps      = criterion.phase_to_stage.unsqueeze(0).expand(N, -1)
            bnd_s  += (probs * (ps != ts.unsqueeze(1)).float()).sum(1).mean().item() * criterion.boundary_penalty
            rv      = torch.tensor(0.0, device=device)
            for ci2, boost in criterion.rare_classes.items():
                m = (lbls == ci2)
                if m.any():
                    rv += ((1 - probs[m, ci2]) * boost).mean()
            rare_s += rv.item()
            n_b    += 1
    ce_v, ord_v, bnd_v, rare_v = (ce_s/n_b, ord_s/n_b, bnd_s/n_b, rare_s/n_b) if n_b else (0,0,0,0)

    # ── Log history ─────────────────────────────────────────────────────────────
    history['train_loss'].append(tr_loss)
    history['val_loss'].append(va_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(va_acc)
    history['train_f1'].append(tr_f1)
    history['val_f1'].append(va_f1)
    history['lr'].append(cur_lr)
    history['ecl_ce'].append(ce_v)
    history['ecl_ordinal'].append(ord_v)
    history['ecl_boundary'].append(bnd_v)
    history['ecl_rare'].append(rare_v)

    elapsed = time.time() - t0
    print(
        f'{epoch:5d}  {tr_loss:7.4f}  {tr_acc:6.4f}  {tr_f1:6.4f}  '
        f'{va_loss:7.4f}  {va_acc:6.4f}  {va_f1:6.4f}  '
        f'{ce_v:6.3f}  {ord_v:6.3f}  {bnd_v:6.3f}  {rare_v:6.3f}  '
        f'{cur_lr:8.1e}  {elapsed:4.0f}s'
    )

    # ── Save checkpoints ────────────────────────────────────────────────────────
    torch.save({
        'epoch': epoch,
        'model_name': MODEL_NAME,
        'model_state_dict': (model.module if hasattr(model, 'module') else model).state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'val_loss': va_loss, 'val_acc': va_acc, 'val_f1': va_f1,
        'history': history,
    }, LATEST_CKPT)

    if va_f1 > best_val_f1:
        best_val_f1 = va_f1
        no_improve  = 0
        torch.save({
            'epoch': epoch,
            'model_name': MODEL_NAME,
            'model_state_dict': (model.module if hasattr(model, 'module') else model).state_dict(),
            'val_acc': va_acc, 'val_f1': va_f1, 'val_loss': va_loss,
        }, BEST_F1_CKPT)
        print(f'  >> Best model saved  val_f1={va_f1:.4f}  epoch={epoch}')
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f'Early stopping: no improvement for {PATIENCE} epochs.')
            break

total_time = time.time() - start_time
print('-' * 110)
print(f'Done.  Total: {total_time/60:.1f} min  |  Best val F1: {best_val_f1:.4f}')


Training VGG16 with ECL Loss
Backbone frozen | 114,055,696 head params | AMP=True
Epoch   TrLoss   TrAcc    TrF1   VaLoss   VaAcc    VaF1      CE     Ord     Bnd    Rare        LR   Time
--------------------------------------------------------------------------------------------------------------


    1   1.3822  0.4356  0.3456   1.4642  0.5034  0.3778   0.571   0.015   0.349   2.524   5.0e-04  1084s
  >> Best model saved  val_f1=0.3778  epoch=1


    2   1.0058  0.5617  0.4690   1.2882  0.5253  0.4117   0.586   0.014   0.325   1.394   4.9e-04   857s
  >> Best model saved  val_f1=0.4117  epoch=2


    3   0.7736  0.6452  0.5691   1.2785  0.5368  0.4284   0.599   0.018   0.360   1.601   4.8e-04   902s
  >> Best model saved  val_f1=0.4284  epoch=3


    4   0.6650  0.6929  0.6281   1.3068  0.5712  0.4725   0.599   0.013   0.321   1.540   4.5e-04   909s
  >> Best model saved  val_f1=0.4725  epoch=4


...

    7   0.4430  0.7969  0.7291   1.4096  0.5750  0.4932   0.605   0.014   0.332   1.775   3.3e-04   977s
  >> Best model saved  val_f1=0.4932  epoch=7


    8   0.3929  0.8135  0.7582   1.4660  0.5709  0.4888   0.775   0.017   0.332   1.516   2.8e-04   959s


    9   0.3485  0.8387  0.7778   1.5159  0.5761  0.4642   0.821   0.018   0.342   1.806   2.2e-04   927s


   10   0.2901  0.8611  0.8208   1.5516  0.5781  0.4643   0.785   0.016   0.334   1.542   1.7e-04   898s


   11   0.2528  0.8741  0.8378   1.6294  0.5707  0.4897   0.886   0.016   0.332   1.554   1.3e-04   931s
Early stopping: no improvement for 4 epochs.
--------------------------------------------------------------------------------------------------------------
Done.  Total: 170.8 min  |  Best val F1: 0.4932


In [16]:
# Cell 9: Training History Plots
ep = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

ax = axes[0, 0]
ax.plot(ep, history['train_loss'], label='Train', linewidth=2)
ax.plot(ep, history['val_loss'],   label='Val',   linewidth=2, linestyle='dashed')
ax.set(title='ECL Loss', xlabel='Epoch', ylabel='Loss')
ax.legend(); ax.grid(alpha=0.3)

ax = axes[0, 1]
ax.plot(ep, [a * 100 for a in history['train_acc']], label='Train', linewidth=2)
ax.plot(ep, [a * 100 for a in history['val_acc']],   label='Val',   linewidth=2, linestyle='dashed')
ax.set(title='Accuracy', xlabel='Epoch', ylabel='Accuracy (%)')
ax.legend(); ax.grid(alpha=0.3)

ax = axes[0, 2]
ax.plot(ep, history['train_f1'], label='Train F1', linewidth=2, color='green')
ax.plot(ep, history['val_f1'],   label='Val F1',   linewidth=2, linestyle='dashed', color='darkgreen')
ax.axhline(best_val_f1, color='red', linestyle='dotted', linewidth=1.5,
           label=f'Best val F1 = {best_val_f1:.4f}')
ax.set(title='Macro F1', xlabel='Epoch', ylabel='Macro F1')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax = axes[1, 0]
ax.plot(ep, history['lr'], linewidth=2, color='orange')
ax.set_yscale('log')
ax.set(title='Learning Rate Schedule', xlabel='Epoch', ylabel='Learning Rate')
ax.grid(alpha=0.3)

ax = axes[1, 1]
ax.plot(ep, history['ecl_ce'],       label='CE (base)',        linewidth=2)
ax.plot(ep, history['ecl_ordinal'],  label='Ordinal (0.3)',    linewidth=2)
ax.plot(ep, history['ecl_boundary'], label='Boundary (0.5)',   linewidth=2)
ax.plot(ep, history['ecl_rare'],     label='Rare FN (0.5)',    linewidth=2)
ax.set(title='ECL Loss Component Breakdown', xlabel='Epoch', ylabel='Component Value')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax = axes[1, 2]
sc = ax.scatter(history['val_acc'], history['val_f1'],
                c=list(ep), cmap='viridis', s=60, zorder=5)
ax.plot(history['val_acc'], history['val_f1'], alpha=0.3, linewidth=1)
plt.colorbar(sc, ax=ax, label='Epoch')
ax.set(title='Val F1 vs Val Accuracy', xlabel='Val Accuracy', ylabel='Val Macro F1')
ax.grid(alpha=0.3)

plt.suptitle(f'{MODEL_DISPLAY} + ECL Loss  —  Training History',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'training_history.png'), dpi=150, bbox_inches='tight')
plt.show()


In [17]:
# Cell 10: Final Evaluation on Test Set

from sklearn.metrics import classification_report, f1_score, confusion_matrix as sk_cm

ckpt = torch.load(BEST_F1_CKPT, map_location=device)
(model.module if hasattr(model, "module") else model).load_state_dict(ckpt['model_state_dict'])
print(f'Loaded best F1 model: epoch {ckpt["epoch"]}  val_f1={ckpt["val_f1"]:.4f}')

print('Running inference on test set...')
test_preds, test_labels = predict(model, test_loader, device)

test_acc    = (test_preds == test_labels).mean()
test_mac_f1 = f1_score(test_labels, test_preds, average='macro',    zero_division=0)
test_wgt_f1 = f1_score(test_labels, test_preds, average='weighted', zero_division=0)
per_cls_f1  = f1_score(test_labels, test_preds, average=None,       zero_division=0)

print()
print(f'Test Results  —  {MODEL_DISPLAY}')
print(f'  Accuracy     : {test_acc:.4f}  ({test_acc * 100:.2f}%)')
print(f'  Macro F1     : {test_mac_f1:.4f}')
print(f'  Weighted F1  : {test_wgt_f1:.4f}')
print()
print(f'  Per-class F1:')
for nm, sc in zip(PHASE_DISPLAY, per_cls_f1):
    status = 'good' if sc >= 0.5 else 'low '
    print(f'    {nm:>5}: {sc:.3f}  [{status}]')

report = classification_report(
    test_labels, test_preds,
    target_names=PHASE_DISPLAY,
    labels=list(range(NUM_CLASSES)),
    zero_division=0,
)
print()
print(report)
with open(os.path.join(PLOTS_DIR, 'classification_report.txt'), 'w') as f:
    f.write(f'Model    : {MODEL_DISPLAY}\n')
    f.write(f'Loss     : ECL\n')
    f.write(f'Accuracy : {test_acc:.4f}\n')
    f.write(f'Macro F1 : {test_mac_f1:.4f}\n\n')
    f.write(report)
print(f'Report saved: {PLOTS_DIR}/classification_report.txt')

Loaded best F1 model: epoch 7  val_f1=0.4932
Running inference on test set...


Inference: 100%|████████████████████████████████| 88/88 [03:12<00:00,  2.19s/it]



Test Results  —  VGG16
  Accuracy     : 0.5740  (57.40%)
  Macro F1     : 0.4549
  Weighted F1  : 0.5875

  Per-class F1:
     pPB2: 0.645  [good]
     pPNa: 0.851  [good]
     pPNf: 0.582  [good]
       p2: 0.705  [good]
       p3: 0.211  [low ]
       p4: 0.592  [good]
       p5: 0.200  [low ]
       p6: 0.168  [low ]
       p7: 0.091  [low ]
       p8: 0.508  [good]
      p9+: 0.632  [good]
       pM: 0.511  [good]
      pSB: 0.564  [good]
       pB: 0.338  [low ]
      pEB: 0.680  [good]
      pHB: 0.000  [low ]

              precision    recall  f1-score   support

        pPB2       0.58      0.72      0.65      1430
        pPNa       0.95      0.77      0.85      6585
        pPNf       0.48      0.73      0.58       999
          p2       0.84      0.61      0.70      4570
          p3       0.14      0.41      0.21       711
          p4       0.74      0.49      0.59      4486
          p5       0.17      0.24      0.20       956
          p6       0.11      0.32      0.17

In [18]:
# Cell 11: Evaluation Plots
plot_confusion_matrix(test_preds, test_labels, save_dir=PLOTS_DIR)
plt.show()

plot_per_class_f1(test_preds, test_labels, save_dir=PLOTS_DIR)
plt.show()

plot_sample_predictions(model, test_loader, device, n_samples=16, save_dir=PLOTS_DIR)
plt.show()

# Top confused class pairs
cm = sk_cm(test_labels, test_preds, labels=list(range(NUM_CLASSES)))
np.fill_diagonal(cm, 0)
confused = sorted(
    [(cm[i, j], i, j) for i in range(NUM_CLASSES) for j in range(NUM_CLASSES) if cm[i, j] > 0],
    reverse=True
)
print('Top 10 Most Confused Class Pairs (True -> Predicted):')
print(f'  {"True":>5}  {"Predicted":>9}  {"Errors":>7}')
print('  ' + '-' * 30)
for cnt, ti, pi in confused[:10]:
    print(f'  {PHASE_DISPLAY[ti]:>5}  {PHASE_DISPLAY[pi]:>9}  {cnt:>7}')


Saved: /kaggle/working/plots/vgg16_ecl/confusion_matrix.png
Saved: /kaggle/working/plots/vgg16_ecl/per_class_f1.png
Saved: /kaggle/working/plots/vgg16_ecl/sample_predictions.png
Top 10 Most Confused Class Pairs (True -> Predicted):
   True  Predicted   Errors
  ------------------------------
     p8        p9+     1225
     p2         p3     1094
    p9+         p8      897
    p9+         pM      848
    p9+        pSB      782
     pB        pEB      731
     p8         p6      700
   pPNa       pPB2      696
     p4         p5      669
     p7         p8      596


---
## Notebook Overview

This notebook classifies 16 embryonic developmental phases from microscopy frames using **VGG16** fine-tuned with the **Embryo Composite Loss (ECL)**.

### Cell Guide

| Cell | Purpose |
|------|--------|
| 0 — GPU Check | Verifies CUDA availability and prints GPU info |
| 1 — Configuration | Sets model name, image size (224x224), batch size (32), epochs, paths, and seed |
| 2 — Dataset EDA | Builds the frame index (297,428 frames, 16 phases, 527:1 imbalance); plots class distribution |
| 3 — Visualization | Displays one representative frame per class and a random 4x4 sample grid |
| 4 — Data Split | Splits 704 embryos at embryo level into Train/Val/Test; verifies no data leakage |
| 5 — DataLoaders | Creates PyTorch DataLoaders with train augmentation and val/test normalization |
| 6 — Model | Loads ImageNet pretrained VGG16 (138M total params); freezes backbone; ~108M head params trained |
| 7 — Loss + Optimizer | Sets up ECL loss (CE + Ordinal + Boundary + Rare-FN) with AdamW and cosine LR schedule |
| 8 — Training | Runs up to 20 epochs; saves `latest_checkpoint.pth` and `best_f1_model.pth`; early stopping after 7 epochs without improvement |
| 9 — Training Plots | Loss curves, accuracy, macro F1, learning rate, ECL components, F1 vs accuracy scatter |
| 10 — Test Evaluation | Loads best model; runs inference on test set; prints accuracy, macro F1, weighted F1, per-class F1 |
| 11 — Eval Plots | Confusion matrix, per-class F1 bar chart, sample predictions grid, top confused pairs |


In [19]:
# Kaggle sessions end automatically when all cells complete.